In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import CamembertTokenizer, CamembertForSequenceClassification
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

# Charger les données

In [ ]:
file_path = "../Data/intent-detection-augmented.csv"
df = pd.read_csv(file_path)

#  Encoder toutes les classes, y compris "out_scope"
label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["label"])
num_classes = len(label_encoder.classes_)

# Split équilibré des classes avec StratifiedShuffleSplit

In [ ]:
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in splitter.split(df["text"], df["label_encoded"]):
    X_train, X_test = df["text"].iloc[train_idx], df["text"].iloc[test_idx]
    y_train, y_test = df["label_encoded"].iloc[train_idx], df["label_encoded"].iloc[test_idx]



# Initialiser le tokenizer
tokenizer = CamembertTokenizer.from_pretrained("camembert-base")

# Définition du dataset

In [ ]:
class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

# Créer les datasets et dataloaders

In [ ]:
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dataset = IntentDataset(X_train, y_train, tokenizer)
test_dataset = IntentDataset(X_test, y_test, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Charger le modèle Camembert

In [ ]:
model = CamembertForSequenceClassification.from_pretrained("camembert-base", num_labels=num_classes).to(dev)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)


# ✅ Entraînement avec Early Stopping

In [ ]:
def train(model, train_loader, optimizer, epochs=300, patience=30):
    model.train()
    best_loss = float("inf")
    patience_counter = 0

    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(dev)
            attention_mask = batch["attention_mask"].to(dev)
            labels = batch["labels"].to(dev)
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

        # Early Stopping: Vérifier si la perte a diminué
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0  # Réinitialiser le compteur
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping déclenché")
                break  # Arrêter l'entraînement si la perte ne diminue plus

train(model, train_loader, optimizer)

# Évaluation

In [ ]:
def evaluate(model, test_loader):
    model.eval()
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(dev)
            attention_mask = batch["attention_mask"].to(dev)
            labels = batch["labels"].to(dev)
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(F.softmax(logits, dim=1), axis=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    print("Accuracy:", accuracy_score(true_labels, predictions))
    print(classification_report(true_labels, predictions, target_names=label_encoder.classes_))

evaluate(model, test_loader)


Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1, Loss: 2.1770
Epoch 2, Loss: 2.0617
Epoch 3, Loss: 1.8687
Epoch 4, Loss: 1.6161
Epoch 5, Loss: 1.3705
Epoch 6, Loss: 1.1577
Epoch 7, Loss: 0.9745
Epoch 8, Loss: 0.8371
Epoch 9, Loss: 0.7159
Epoch 10, Loss: 0.6125
Epoch 11, Loss: 0.5245
Epoch 12, Loss: 0.4562
Epoch 13, Loss: 0.3960
Epoch 14, Loss: 0.3454
Epoch 15, Loss: 0.3042
Epoch 16, Loss: 0.2655
Epoch 17, Loss: 0.2356
Epoch 18, Loss: 0.2093
Epoch 19, Loss: 0.1921
Epoch 20, Loss: 0.1793
Epoch 21, Loss: 0.1524
Epoch 22, Loss: 0.1367
Epoch 23, Loss: 0.1175
Epoch 24, Loss: 0.1129
Epoch 25, Loss: 0.1006
Epoch 26, Loss: 0.0945
Epoch 27, Loss: 0.0855
Epoch 28, Loss: 0.0788
Epoch 29, Loss: 0.0762
Epoch 30, Loss: 0.0685
Epoch 31, Loss: 0.0621
Epoch 32, Loss: 0.0583
Epoch 33, Loss: 0.0549
Epoch 34, Loss: 0.0526
Epoch 35, Loss: 0.0485
Epoch 36, Loss: 0.0456
Epoch 37, Loss: 0.0432
Epoch 38, Loss: 0.0392
Epoch 39, Loss: 0.0378
Epoch 40, Loss: 0.0361
Epoch 41, Loss: 0.0340
Epoch 42, Loss: 0.0322
Epoch 43, Loss: 0.0306
Epoch 44, Loss: 0.02

#  Fonction de prédiction avec gestion de "out_scope"

In [ ]:


def predict_with_threshold(text, threshold=0.5):
    model.eval()
    encoding = tokenizer(text, truncation=True, padding="max_length", max_length=128, return_tensors="pt")
    input_ids = encoding["input_ids"].to(dev)
    attention_mask = encoding["attention_mask"].to(dev)

    with torch.no_grad():
        logits = model(input_ids, attention_mask=attention_mask).logits
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]
        max_prob = np.max(probs)
        predicted_label = np.argmax(probs)

        # 🔹 Si aucune classe n'a une forte probabilité, renvoyer "out_scope"
        if max_prob < threshold:
            return "out_scope"

        return label_encoder.inverse_transform([predicted_label])[0]




In [17]:
texte_test = "je veux aller à casablanca"
print("Prédiction :", predict_with_threshold(texte_test))

Prédiction : travel_suggestion


In [18]:
model.save_pretrained("saved_model")
tokenizer.save_pretrained("saved_model")

('saved_model/tokenizer_config.json',
 'saved_model/special_tokens_map.json',
 'saved_model/sentencepiece.bpe.model',
 'saved_model/added_tokens.json')

# Hebergement sur HuggingFace ✅